In [ ]:
!pip install pennylane==0.33 pennylane-qiskit==0.33 qiskit qiskit-ibm-provider

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 📌 Load Data
df = pd.read_csv("/content/drive/MyDrive/QML/parkinson_gait/final_parkinsons_for_the_love_of_god.csv")

df.head()

,Unnamed: 0,Gait_Cycle_Percent,Left_Ankle_Ankle_Dorsi_Plantarflexion_PF_DF,Left_Hip_Hip_Flexion_Extension_EXT_FLX,Left_Knee_Knee_Flx_Extension_EXT_FLX,Right_Ankle_Ankle_Dorsi_Plantarflexion_PF_DF,Right_Hip_Hip_Flexion_Extension_EXT_FLX,Right_Knee_Knee_Flx_Extension_EXT_FLX,Subject_ID,label,...,Left_Knee_Ang_Knee_Flx_Extension_Variability,Right_Ankle_Ang_Ankle_Dorsi_Plantarflexion_Variability,Right_Hip_Ang_Hip_Flexion_Extension_Variability,Right_Knee_Ang_Knee_Flx_Extension_Variability,Left_Ankle_Ang_Ankle_Dorsi_Plantarflexion_Acceleration,Left_Hip_Ang_Hip_Flexion_Extension_Acceleration,Left_Knee_Ang_Knee_Flx_Extension_Acceleration,Right_Ankle_Ang_Ankle_Dorsi_Plantarflexion_Acceleration,Right_Hip_Ang_Hip_Flexion_Extension_Acceleration,Right_Knee_Ang_Knee_Flx_Extension_Acceleration
0,0,1,-2.37648,34.02794,16.14490,1.87055,33.60558,11.36783,SUB01_1,1,...,18.945746,7.692659,15.308019,18.810668,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
1,1,2,-1.87879,33.60433,16.86324,0.98626,33.50635,11.34289,SUB01_1,1,...,18.945746,7.692659,15.308019,18.810668,0.49769,-0.42361,0.71834,-0.88429,-0.09923,-0.02494
2,2,3,-1.15549,33.26338,17.86806,0.16397,33.50398,11.92080,SUB01_1,1,...,18.945746,7.692659,15.308019,18.810668,0.72330,-0.34095,1.00482,-0.82229,-0.00237,0.57791
3,3,4,-0.21892,33.00112,19.09299,-0.47002,33.58014,13.01924,SUB01_1,1,...,18.945746,7.692659,15.308019,18.810668,0.93657,-0.26226,1.22493,-0.63399,0.07616,1.09844
4,4,5,0.88757,32.78223,20.42713,-0.80742,33.70621,14.51952,SUB01_1,1,...,18.945746,7.692659,15.308019,18.810668,1.10649,-0.21889,1.33414,-0.33740,0.12607,1.50028


# Step 1: Quantum Feature Extraction

In [ ]:
from google.colab import userdata
QIBM_API_KEY = userdata.get('QIBM_API_KEY')

In [ ]:
from qiskit_ibm_provider import IBMProvider

IBMProvider.save_account(QIBM_API_KEY, overwrite=True)
provider = IBMProvider()

In [ ]:
from qiskit_ibm_provider import least_busy

backend = least_busy(
    provider.backends(
        simulator=False,
        min_num_qubits=4,
        operational=True,
        dynamic_reprate_enabled=True
    )
)

print(f"Running on backend: {backend.name}")

Running on backend: ibm_sherbrooke


In [ ]:
from qiskit.primitives import Sampler
from qiskit import QuantumCircuit

# Define quantum circuit
num_qubits = 3
features = np.random.uniform(0, np.pi, num_qubits)

qc = QuantumCircuit(num_qubits, num_qubits)

for i in range(num_qubits):
    qc.ry(features[i], i)

for i in range(num_qubits - 1):
    qc.cx(i, i + 1)
qc.cx(num_qubits - 1, 0)

# ✅ Explicitly map qubit i → classical bit i
qc.measure(range(num_qubits), range(num_qubits))

sampler = Sampler()
job = sampler.run([qc])
result = job.result()

print("Quantum output:", result.quasi_dists[0])

Quantum output: {0: np.float64(0.431097250921165), 1: np.float64(0.106352209668219), 2: np.float64(0.007315577337737), 3: np.float64(0.005187449260214), 4: np.float64(0.003059756499286), 5: np.float64(0.012402681800834), 6: np.float64(0.180307384425602), 7: np.float64(0.254277690086943)}


<ipython-input-26-3e425b53d8ea>:20: DeprecationWarning: The class ``qiskit.primitives.sampler.Sampler`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseSamplerV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Sampler` class is `StatevectorSampler`.
  sampler = Sampler()


In [ ]:
def quantum_feature_vector(features, backend_name="ibm_kyiv", num_qubits=5):
    from qiskit_ibm_provider import IBMProvider
    from qiskit.primitives import Sampler
    from qiskit import QuantumCircuit
    import numpy as np

    provider = IBMProvider()
    qc = QuantumCircuit(num_qubits, num_qubits)

    for i in range(num_qubits):
        qc.ry(features[i], i)

    for i in range(num_qubits - 1):
        qc.cx(i, i + 1)
    qc.cx(num_qubits - 1, 0)

    qc.measure(range(num_qubits), range(num_qubits))

    sampler = Sampler()
    job = sampler.run([qc])
    result = job.result()

    vec = [float(result.quasi_dists[0].get(i, 0.0)) for i in range(2 ** num_qubits)]
    return vec

In [ ]:
import hashlib
import json
import os

# --- CONFIG ---
CACHE_FILE = "/content/drive/MyDrive/QML/parkinson_gait/quantum_cache_parkinson_windowed.json"
num_qubits = 5

# --- Hashing utility ---
def hash_input(arr):
    return hashlib.md5(np.round(arr, 6).tobytes()).hexdigest()

# --- Load existing cache ---
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r") as f:
        cache = json.load(f)
else:
    cache = {}

# --- Safe quantum feature wrapper ---
def cached_quantum_feature_vector(features):
    key = hash_input(features)
    if key in cache:
        return cache[key]

    try:
        vec = quantum_feature_vector(features, num_qubits=num_qubits)
        cache[key] = vec

        # Save after each success
        with open(CACHE_FILE, "w") as f:
            json.dump(cache, f)

        print(f"✅ Processed: {key}")
        return vec

    except Exception as e:
        print(f"❌ Error on input {key}: {e}")
        return [0.0] * (2 ** num_qubits)  # placeholder if fail

# Step 2: Apply Quantum Encoding to Entire Dataset

In [ ]:
def extract_quantum_features_deduplicated(
    df,
    gait_features,
    quantum_feature_fn,
    num_qubits=5,
    cache_file="/content/drive/MyDrive/QML/parkinson_gait/quantum_cache.json",
    save_every=10,
    output_csv="/content/drive/MyDrive/QML/parkinson_gait/parkinsons_features_quantum_hardware.csv"
):
    import numpy as np
    import pandas as pd
    import os
    import json
    import hashlib
    from tqdm import tqdm

    # Step 1: Round + deduplicate
    X_classical = df[gait_features].values
    X_classical_rounded = np.round(X_classical, 6)
    X_unique, idx_unique, idx_inverse = np.unique(
        X_classical_rounded, axis=0, return_index=True, return_inverse=True
    )

    print(f"🔍 Unique samples: {len(X_unique)} / {len(X_classical)} total")

    # Step 2: Load or initialize cache
    if os.path.exists(cache_file):
        with open(cache_file, "r") as f:
            cache = json.load(f)
    else:
        cache = {}

    # Step 3: Hashing function
    def hash_input(arr):
        return hashlib.md5(np.round(arr, 6).tobytes()).hexdigest()

    # Step 4: Safe call with cache
    def cached_quantum_call(features):
        key = hash_input(features)
        if key in cache:
            return cache[key]
        try:
            vec = quantum_feature_fn(features)
            cache[key] = vec
            with open(cache_file, "w") as f:
                json.dump(cache, f)
            return vec
        except Exception as e:
            print(f"❌ Error processing {key}: {e}")
            return [0.0] * (2 ** num_qubits)

    # Step 5: Extract features with incremental saving
    X_quantum_unique = []
    for i, x in enumerate(tqdm(X_unique)):
        X_quantum_unique.append(cached_quantum_call(x))

        # Save progress every N samples
        if (i + 1) % save_every == 0 or (i + 1) == len(X_unique):
            print(f"💾 Saving progress at sample {i + 1}")
            processed_mask = np.isin(idx_inverse, range(i + 1))

            # Reconstruct partial quantum features
            partial_full = np.array(X_quantum_unique)[idx_inverse[processed_mask]]

            partial_df = pd.DataFrame(
                partial_full,
                columns=[f"Quantum_Feature_{j}" for j in range(2 ** num_qubits)]
            )
            partial_df["label"] = df["label"].values[processed_mask]
            partial_df["Subject_ID"] = df["Subject_ID"].values[processed_mask]

            # Save it
            partial_df.to_csv(output_csv, index=False)

    # Step 6: Reconstruct full dataset
    X_quantum_full = np.array(X_quantum_unique)[idx_inverse]

    # Step 7: Final DataFrame
    df_quantum = pd.DataFrame(
        X_quantum_full,
        columns=[f"Quantum_Feature_{i}" for i in range(2 ** num_qubits)]
    )
    df_quantum["label"] = df["label"].values
    df_quantum["Subject_ID"] = df["Subject_ID"].values

    # Final save
    df_quantum.to_csv(output_csv, index=False)
    print(f"✅ Final full CSV saved: {output_csv}")

    return df_quantum

In [ ]:
X_classical = df.drop(columns=["label", "Subject_ID", "Gait_Cycle_Percent", "Unnamed: 0"], errors="ignore")

gait_features = X_classical.columns.tolist()

In [ ]:
gait_features

['Left_Ankle_Ankle_Dorsi_Plantarflexion_PF_DF',
 'Left_Hip_Hip_Flexion_Extension_EXT_FLX',
 'Left_Knee_Knee_Flx_Extension_EXT_FLX',
 'Right_Ankle_Ankle_Dorsi_Plantarflexion_PF_DF',
 'Right_Hip_Hip_Flexion_Extension_EXT_FLX',
 'Right_Knee_Knee_Flx_Extension_EXT_FLX',
 'Left_Ankle_Ang_Ankle_Dorsi_Plantarflexion_Variability',
 'Left_Hip_Ang_Hip_Flexion_Extension_Variability',
 'Left_Knee_Ang_Knee_Flx_Extension_Variability',
 'Right_Ankle_Ang_Ankle_Dorsi_Plantarflexion_Variability',
 'Right_Hip_Ang_Hip_Flexion_Extension_Variability',
 'Right_Knee_Ang_Knee_Flx_Extension_Variability',
 'Left_Ankle_Ang_Ankle_Dorsi_Plantarflexion_Acceleration',
 'Left_Hip_Ang_Hip_Flexion_Extension_Acceleration',
 'Left_Knee_Ang_Knee_Flx_Extension_Acceleration',
 'Right_Ankle_Ang_Ankle_Dorsi_Plantarflexion_Acceleration',
 'Right_Hip_Ang_Hip_Flexion_Extension_Acceleration',
 'Right_Knee_Ang_Knee_Flx_Extension_Acceleration']

In [ ]:
df_quantum = extract_quantum_features_deduplicated(
    df=df,
    gait_features=gait_features,
    quantum_feature_fn=quantum_feature_vector,
    num_qubits=5,
    cache_file="/content/drive/MyDrive/QML/parkinson_gait/quantum_cache.json",
    save_every=10,
    output_csv="/content/drive/MyDrive/QML/parkinson_gait/parkinsons_features_quantum_hardware_please.csv"
)


🔍 Unique samples: 4848 / 4848 total


  0%|          | 0/4848 [00:00<?, ?it/s]<ipython-input-27-84eef1b67573>:19: DeprecationWarning: The class ``qiskit.primitives.sampler.Sampler`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseSamplerV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Sampler` class is `StatevectorSampler`.
  sampler = Sampler()
  0%|          | 10/4848 [00:33<4:34:18,  3.40s/it]

💾 Saving progress at sample 10


<ipython-input-27-84eef1b67573>:19: DeprecationWarning: The class ``qiskit.primitives.sampler.Sampler`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseSamplerV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Sampler` class is `StatevectorSampler`.
  sampler = Sampler()
  0%|          | 20/4848 [01:04<4:16:30,  3.19s/it]

💾 Saving progress at sample 20


...

💾 Saving progress at sample 4830


<ipython-input-27-84eef1b67573>:19: DeprecationWarning: The class ``qiskit.primitives.sampler.Sampler`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseSamplerV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Sampler` class is `StatevectorSampler`.
  sampler = Sampler()
100%|█████████▉| 4840/4848 [4:13:30<00:25,  3.22s/it]

💾 Saving progress at sample 4840


<ipython-input-27-84eef1b67573>:19: DeprecationWarning: The class ``qiskit.primitives.sampler.Sampler`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseSamplerV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Sampler` class is `StatevectorSampler`.
  sampler = Sampler()
100%|██████████| 4848/4848 [4:13:55<00:00,  3.14s/it]

💾 Saving progress at sample 4848
✅ Final full CSV saved: /content/drive/MyDrive/QML/parkinson_gait/parkinsons_features_quantum_hardware_please.csv


In [ ]:
df[gait_features]

,ForcePlate_2_mean,ForcePlate_2_std,ForcePlate_2_max,ForcePlate_2_range,ForcePlate_2_iqr,ForcePlate_2_energy,ForcePlate_2_rms,ForcePlate_2_skew,ForcePlate_2_kurtosis,ForcePlate_2_dominant_freq,...,ForcePlate_19_mean,ForcePlate_19_std,ForcePlate_19_max,ForcePlate_19_range,ForcePlate_19_iqr,ForcePlate_19_energy,ForcePlate_19_rms,ForcePlate_19_skew,ForcePlate_19_kurtosis,ForcePlate_19_dominant_freq
0,172.359367,143.888911,418.88,414.26,284.6800,1.512353e+07,224.525656,0.148185,-1.499030,0.006667,...,634.180433,422.240115,1131.79,1131.79,978.0650,1.741415e+08,761.886827,-0.449844,-1.370928,0.006667
1,145.777133,152.755170,432.41,432.41,274.0925,1.337553e+07,211.151876,0.515931,-1.335367,0.006667,...,491.962167,483.780238,1158.30,1158.30,1000.9450,1.428210e+08,689.978327,0.136567,-1.853274,0.006667
2,114.031500,136.622165,432.41,432.41,236.9400,9.500640e+06,177.957295,0.821438,-0.837736,0.006667,...,517.942333,492.622034,1220.45,1220.45,1012.1375,1.532822e+08,714.801181,0.071004,-1.834301,0.006667
3,101.729833,132.147795,368.17,368.17,231.7700,8.343600e+06,166.769298,0.869513,-0.937864,0.006667,...,575.117033,487.173268,1220.45,1220.45,1031.6075,1.704292e+08,753.722360,-0.172900,-1.820402,0.006667
4,96.336533,127.534214,376.09,376.09,213.6200,7.663711e+06,159.830233,0.911398,-0.770219,0.006667,...,623.639867,476.932279,1115.95,1115.95,1029.5450,1.849173e+08,785.105778,-0.413489,-1.718276,0.006667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5913,69.024633,86.291361,250.80,250.80,147.5100,3.663180e+06,110.501579,0.873116,-0.863150,0.010000,...,495.266933,423.163922,1065.68,1065.68,934.4500,1.273071e+08,651.426926,-0.035448,-1.715901,0.010000
5914,69.532833,90.676881,250.80,250.80,156.2275,3.917133e+06,114.267719,0.874999,-0.916010,0.006667,...,577.026633,399.735854,1059.30,1059.30,882.2550,1.478245e+08,701.960461,-0.432189,-1.472263,0.006667
5915,87.687233,93.763869,244.42,244.42,193.2700,4.944214e+06,128.377233,0.435041,-1.554129,0.010000,...,511.219500,405.442885,1058.86,1058.86,883.9600,1.277188e+08,652.479356,-0.109681,-1.654724,0.010000
5916,86.758467,90.207240,234.96,234.96,190.5200,4.699313e+06,125.157412,0.459379,-1.498940,0.006667,...,428.646167,409.161572,1058.86,1058.86,840.6200,1.053452e+08,592.579723,0.240478,-1.620321,0.006667
